<a href="https://colab.research.google.com/github/OJB-Quantum/Notebooks-for-Ideas/blob/main/Fibonacci_Password_Generator_in_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Authored by Onri Jay Benally (2026)

Open Access (CC-BY-4.0)

In [ ]:
"""Generate reproducible strings with Fibonacci-ordered internal symbols.

Mnemonic mode preserves a vowel-substituted phrase. Derived mode replaces
that base with a scrypt/HMAC-derived alphanumeric string before interleaving.
Use demonstration secrets in hosted notebooks. This custom construction has
received functional testing only; secret strength requires a separate review.
"""

import dataclasses
import getpass
import hashlib
import hmac
import json
import string
import warnings
from collections.abc import Callable, Iterator


# --- Control knobs -----------------------------------------------------------
DEFAULT_MODE = "derived"  # Choose "mnemonic" for the literal phrase recipe.
DEFAULT_PRIMARY = "!@#$%&*?"
DEFAULT_MODIFIERS = "+^-"
DEFAULT_VOWEL_DIGITS = "43107"  # Corresponds to a, e, i, o, u.
DEFAULT_FIBONACCI_SKIP = 0  # Zero starts with 1, 1, 2, 3, 5, ...
DEFAULT_TOTAL_LENGTH = 40  # Used only in derived mode.
MAX_OUTPUT_LENGTH = 256
MIN_DERIVED_BASE_LENGTH = 24
MIN_MASTER_LENGTH = 20  # An input guard, independent of entropy.
MAX_SECRET_BYTES = 1024
SCRYPT_N = 2**17
SCRYPT_R = 8
SCRYPT_P = 1
SCRYPT_MAXMEM = 256 * 1024**2
MAX_POLICY_ATTEMPTS = 1000

# Preserve these definitions to reproduce existing passwords.
SCHEME_ID = "fibonacci-interleave-v1"
ALPHABET = string.ascii_lowercase + string.ascii_uppercase + string.digits
RESERVED_MODIFIERS = "+^-"
PRIMARY_ALLOWED = "".join(
    char for char in string.punctuation if char not in RESERVED_MODIFIERS
)


@dataclasses.dataclass(frozen=True)
class Recipe:
    """Store public formatting and account-specific derivation settings."""

    mode: str = DEFAULT_MODE
    primary: str = DEFAULT_PRIMARY
    modifiers: str = DEFAULT_MODIFIERS
    vowel_digits: str = DEFAULT_VOWEL_DIGITS
    fib_skip: int = DEFAULT_FIBONACCI_SKIP
    total_length: int = DEFAULT_TOTAL_LENGTH
    service: str = ""
    account: str = ""
    revision: int = 1


def ask_text(
    label: str,
    default: str,
    accepts: Callable[[str], bool],
    guidance: str,
) -> str:
    """Prompt until a stripped, public input satisfies its acceptance rule."""
    suffix = f" [{default}]" if default else ""
    while True:
        value = input(f"{label}{suffix}: ").strip() or default
        if accepts(value):
            return value
        print(guidance)


def ask_integer(label: str, default: int, lower: int, upper: int) -> int:
    """Read a bounded integer, accepting Enter for the supplied default."""
    if lower > upper or not lower <= default <= upper:
        raise ValueError("The integer prompt has inconsistent bounds.")
    while True:
        raw = input(f"{label} [{default}]: ").strip()
        try:
            value = int(raw) if raw else default
        except ValueError:
            print("Enter a whole number.")
            continue
        if lower <= value <= upper:
            return value
        print(f"Choose a value from {lower} to {upper}.")


def read_secret(label: str, minimum: int) -> str:
    """Read and confirm a secret, aborting if echo-free input is unavailable.

    Args:
        label: Description displayed beside the hidden input.
        minimum: Minimum character count, used as an input guard.

    Returns:
        The confirmed string, preserving case and whitespace exactly.

    Raises:
        RuntimeError: The environment would echo secret input.
    """
    while True:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("error", getpass.GetPassWarning)
                value = getpass.getpass(f"{label}: ")
                if (
                    len(value) < minimum
                    or len(value.encode()) > MAX_SECRET_BYTES
                ):
                    print(
                        f"Use at least {minimum} characters and at most "
                        f"{MAX_SECRET_BYTES} UTF-8 bytes."
                    )
                    continue
                repeated = getpass.getpass(f"Confirm {label.lower()}: ")
        except getpass.GetPassWarning:
            raise RuntimeError(
                "Hidden input is unavailable. Use a trusted local terminal."
            ) from None
        if hmac.compare_digest(value.encode(), repeated.encode()):
            return value
        print("The entries differ. Enter the secret again.")


def normalize_phrase(phrase: str) -> str:
    """Normalize whitespace in an ASCII phrase containing two or more words."""
    if len(phrase.encode()) > MAX_SECRET_BYTES:
        raise ValueError("The phrase exceeds the input size limit.")
    allowed = ALPHABET + string.whitespace
    if any(char not in allowed for char in phrase):
        raise ValueError("Use ASCII letters, digits, and spaces.")
    words = phrase.split()
    if len(words) < 2 or not any(char.isalpha() for char in phrase):
        raise ValueError("Use two or more words, including letters.")
    return " ".join(words)


def validate_recipe(recipe: Recipe) -> None:
    """Reject ambiguous symbols and unsupported formatting parameters."""
    if recipe.mode not in {"mnemonic", "derived"}:
        raise ValueError("Choose mnemonic or derived mode.")
    if not recipe.primary or len(set(recipe.primary)) != len(recipe.primary):
        raise ValueError("Primary symbols must be nonempty and unique.")
    if any(char not in PRIMARY_ALLOWED for char in recipe.primary):
        raise ValueError("Use ASCII punctuation, reserving +, ^, and -.")
    if len(set(recipe.modifiers)) != len(recipe.modifiers):
        raise ValueError("Each modifier may appear only once.")
    if any(char not in RESERVED_MODIFIERS for char in recipe.modifiers):
        raise ValueError("Modifiers must come from +, ^, and -.")
    if len(recipe.vowel_digits) != 5 or any(
        char not in string.digits for char in recipe.vowel_digits
    ):
        raise ValueError("Supply five ASCII digits for a, e, i, o, u.")
    if not 0 <= recipe.fib_skip <= 10_000:
        raise ValueError("Fibonacci skip must be between 0 and 10000.")
    if recipe.mode == "derived":
        inserted = len(recipe.primary) + len(recipe.modifiers)
        base_length = recipe.total_length - inserted
        if base_length < max(MIN_DERIVED_BASE_LENGTH, inserted + 1):
            raise ValueError("Increase the total length for enough gaps.")
        if recipe.total_length > MAX_OUTPUT_LENGTH:
            raise ValueError("The requested length exceeds the output limit.")
        if not recipe.service.strip() or not recipe.account.strip():
            raise ValueError("Supply both service and account labels.")
        if not 1 <= recipe.revision <= 1_000_000:
            raise ValueError("Revision must be between 1 and 1000000.")


def fibonacci_permutation(size: int, skip: int = 0) -> list[int]:
    """Order indices by Fibonacci steps over a shrinking circular pool.

    Args:
        size: Number of indices to permute.
        skip: Number of Fibonacci terms to discard before selection.

    Returns:
        Every index from zero to size minus one, appearing exactly once.
    """
    if size < 0 or skip < 0:
        raise ValueError("Size and skip must be nonnegative.")
    first, second = 1, 1
    for _ in range(skip):
        first, second = second, first + second
    remaining = list(range(size))
    order = []
    cursor = 0
    while remaining:
        cursor = (cursor + first - 1) % len(remaining)
        order.append(remaining.pop(cursor))
        first, second = second, first + second
    return order


def interleave(base: str, recipe: Recipe) -> str:
    """Insert symbols into distinct gaps, preserving base character order.

    Modifier gaps use evenly spaced midpoints and are reserved first.
    Fibonacci selection then orders primary symbols and selects free gaps.
    Sorting the selected gaps preserves the primary symbols' reading order.
    """
    validate_recipe(recipe)
    if not base or any(char not in ALPHABET for char in base):
        raise ValueError("The foundation must contain ASCII alphanumerics.")
    gap_count = len(base) - 1
    inserted = len(recipe.primary) + len(recipe.modifiers)
    if inserted > gap_count:
        raise ValueError("Use a longer phrase or fewer symbols.")
    if len(base) + inserted > MAX_OUTPUT_LENGTH:
        raise ValueError("The output is too long; truncation is disabled.")

    placements = {}
    modifier_count = len(recipe.modifiers)
    for index, modifier in enumerate(recipe.modifiers):
        gap = ((2 * index + 1) * gap_count) // (2 * modifier_count)
        placements[gap] = modifier

    free_gaps = [gap for gap in range(gap_count) if gap not in placements]
    gap_order = fibonacci_permutation(len(free_gaps), recipe.fib_skip)
    chosen_gaps = sorted(
        free_gaps[index] for index in gap_order[:len(recipe.primary)]
    )
    symbol_order = fibonacci_permutation(len(recipe.primary), recipe.fib_skip)
    for gap, index in zip(chosen_gaps, symbol_order):
        placements[gap] = recipe.primary[index]
    return "".join(
        char + placements.get(index, "") for index, char in enumerate(base)
    )


def keyed_bytes(key: bytes) -> Iterator[int]:
    """Yield a reproducible HMAC-SHA-256 byte stream with a domain label."""
    for counter in range(2**64):
        message = (
            b"fibonacci-interleave-v1/base\x00" + counter.to_bytes(8, "big")
        )
        yield from hmac.digest(key, message, "sha256")
    raise RuntimeError("The derivation counter is exhausted.")


def derive_base(phrase: str, master: str, recipe: Recipe) -> str:
    """Derive an account-specific base using scrypt and rejection sampling.

    The context-derived salt is public domain separation for regeneration.
    Lowercase, uppercase, and digit requirements use whole-candidate rejection.
    Input entropy limits security; output length provides no entropy estimate.
    """
    if (
        len(master) < MIN_MASTER_LENGTH
        or len(master.encode()) > MAX_SECRET_BYTES
    ):
        raise ValueError("The master secret is outside the length range.")
    if not callable(getattr(hashlib, "scrypt", None)):
        raise RuntimeError("This Python build lacks scrypt support.")
    if (
        SCRYPT_N < 2**17 or SCRYPT_N & (SCRYPT_N - 1)
        or SCRYPT_R < 8 or SCRYPT_P < 1
    ):
        raise ValueError("Use power-of-two N >= 2**17, r >= 8, and p >= 1.")
    context = {
        "scheme": SCHEME_ID,
        "phrase": phrase,
        "recipe": dataclasses.asdict(recipe),
        "alphabet": ALPHABET,
        "scrypt": [SCRYPT_N, SCRYPT_R, SCRYPT_P, 32],
    }
    encoded = json.dumps(
        context, sort_keys=True, separators=(",", ":")
    ).encode()
    salt = hashlib.sha256(encoded).digest()
    key = hashlib.scrypt(
        master.encode(), salt=salt, n=SCRYPT_N, r=SCRYPT_R, p=SCRYPT_P,
        maxmem=SCRYPT_MAXMEM, dklen=32,
    )
    stream = keyed_bytes(key)
    inserted = len(recipe.primary) + len(recipe.modifiers)
    base_length = recipe.total_length - inserted
    cutoff = 256 - (256 % len(ALPHABET))
    required_groups = (
        string.ascii_lowercase, string.ascii_uppercase, string.digits
    )
    for _ in range(MAX_POLICY_ATTEMPTS):
        characters = []
        while len(characters) < base_length:
            byte = next(stream)
            if byte < cutoff:  # Reject the remainder to avoid modulo bias.
                characters.append(ALPHABET[byte % len(ALPHABET)])
        candidate = "".join(characters)
        if all(
            any(char in group for char in candidate)
            for group in required_groups
        ):
            return candidate
    raise RuntimeError("Character-policy sampling exceeded its attempt limit.")


def generate_password(phrase: str, recipe: Recipe, master: str = "") -> str:
    """Generate a password from a phrase, recipe, and optional master secret.

    Args:
        phrase: ASCII phrase whose whitespace will be normalized.
        recipe: Formatting settings and, in derived mode, account context.
        master: Independent secret used exclusively in derived mode.

    Returns:
        A reproducible ASCII authentication string.

    Raises:
        ValueError: Inputs violate the recipe or output limits.
        RuntimeError: A required cryptographic capability is unavailable.
    """
    validate_recipe(recipe)
    normalized = normalize_phrase(phrase)
    if recipe.mode == "derived":
        base = derive_base(normalized, master, recipe)
    else:
        substitutions = dict(zip("aeiou", recipe.vowel_digits))
        base = "".join(
            substitutions.get(char.lower(), char)
            for char in normalized if char != " "
        )
    return interleave(base, recipe)


def clear_display() -> None:
    """Clear notebook output where supported; secure erasure is separate."""
    try:
        from IPython.display import clear_output
    except ImportError:
        return
    clear_output(wait=False)


def main() -> None:
    """Collect inputs and reveal the generated string by explicit request."""
    print("Use demo secrets in Colab; prefer local execution for real ones.")
    print("Mnemonic preserves the phrase; derived uses a separate secret.")
    mode = ask_text(
        "Mode", DEFAULT_MODE, lambda value: value in {"mnemonic", "derived"},
        "Enter mnemonic or derived.",
    )
    if mode == "mnemonic":
        print("Mnemonic mode adds zero randomness; its base is recoverable.")
    while True:
        try:
            phrase = normalize_phrase(read_secret("Workflow phrase", 3))
            break
        except ValueError as error:
            print(error)
    primary = ask_text(
        "Primary symbols, in starting order", DEFAULT_PRIMARY,
        lambda value: bool(value) and len(set(value)) == len(value)
        and all(char in PRIMARY_ALLOWED for char in value),
        "Use unique ASCII punctuation; reserve +, ^, and - for modifiers.",
    )
    modifiers = ask_text(
        "Modifier order (enter none to omit)", DEFAULT_MODIFIERS,
        lambda value: value == "none" or (
            len(set(value)) == len(value)
            and all(char in RESERVED_MODIFIERS for char in value)
        ),
        "Use a unique subset of +^- in your desired order, or enter none.",
    )
    vowel_digits = ask_text(
        "Five vowel digits for a, e, i, o, u", DEFAULT_VOWEL_DIGITS,
        lambda value: len(value) == 5
        and all(char in string.digits for char in value),
        "Enter exactly five ASCII digits, such as 43107.",
    )
    recipe = Recipe(
        mode=mode,
        primary=primary,
        modifiers="" if modifiers == "none" else modifiers,
        vowel_digits=vowel_digits,
        fib_skip=ask_integer(
            "Fibonacci terms to skip", DEFAULT_FIBONACCI_SKIP, 0, 10_000
        ),
    )
    master = ""
    if mode == "derived":
        service = ask_text(
            "Service label", "", bool, "Enter a stable service label."
        )
        account = ask_text(
            "Account label", "", bool, "Enter a stable account label."
        )
        revision = ask_integer("Password revision", 1, 1, 1_000_000)
        inserted = len(recipe.primary) + len(recipe.modifiers)
        minimum = inserted + max(MIN_DERIVED_BASE_LENGTH, inserted + 1)
        length = ask_integer(
            "Total password length", max(DEFAULT_TOTAL_LENGTH, minimum),
            minimum, MAX_OUTPUT_LENGTH,
        )
        recipe = dataclasses.replace(
            recipe, service=service, account=account, revision=revision,
            total_length=length,
        )
        print("Use a separate random master secret or a random passphrase.")
        master = read_secret("Master secret", MIN_MASTER_LENGTH)

    password = generate_password(phrase, recipe, master)
    print(f"Length: {len(password)} characters. Entropy is unmeasured.")
    print("Keep the same normalized inputs, recipe, and algorithm.")
    try:
        reveal = input("Type REVEAL to show the password; Enter discards: ")
        if reveal == "REVEAL":
            print("\n" + password + "\n")
            input("Press Enter after storing the password securely: ")
    finally:
        # Clearing references and output leaves possible memory copies.
        phrase = master = password = ""
        clear_display()
    print("Finished. Treat saved output and copied text as sensitive.")


if __name__ == "__main__":
    try:
        main()
    except (EOFError, KeyboardInterrupt):
        clear_display()
        print("Session cancelled.")
    except (ValueError, RuntimeError, MemoryError, OSError) as error:
        clear_display()
        print(f"Generation stopped: {error}")

Use demo secrets in Colab; prefer local execution for real ones.
Mnemonic preserves the phrase; derived uses a separate secret.
